# 10c -- MTMC Stages 4-5: Association + Evaluation

**Prerequisite**: attach **10b's output** as a data source:
`Add Data -> Kernel Output -> search "mtmc-10b-stage-3-faiss-indexing" -> add`

**This is the iteration loop** -- edit the tuning params cell and re-run in ~6 min.
No GPU needed, no data download, no model inference.

| Stage | What | Time |
|---|---|---|
| 4 | Cross-camera association (AQE + Louvain graph clustering) | ~5 min |
| 5 | Evaluation: IDF1, MOTA, HOTA | ~1 min |

In [ ]:
import os, sys, subprocess, shutil, json, time, tarfile
from pathlib import Path

# --- Guard: detect GPU BEFORE importing torch ---
# Kaggle's PyTorch 2.10+ drops P100 (sm_60) support.
# If we got a P100, downgrade to a compatible build first.
if shutil.which("nvidia-smi"):
    _nvsmi = subprocess.run(
        ["nvidia-smi", "--query-gpu=gpu_name,compute_cap", "--format=csv,noheader"],
        capture_output=True, text=True)
    if _nvsmi.returncode == 0 and _nvsmi.stdout.strip():
        _gpu_name, _cap = _nvsmi.stdout.strip().split(",", 1)
        _major, _minor = _cap.strip().split(".")
        _sm = int(_major) * 10 + int(_minor)
        if _sm < 70:
            print(f"\u26a0 GPU {_gpu_name.strip()} (sm_{_sm}) — installing compatible PyTorch ...")
            subprocess.check_call([
                sys.executable, "-m", "pip", "install", "-q",
                "torch==2.4.1", "torchvision==0.19.1",
                "--index-url", "https://download.pytorch.org/whl/cu124",
            ])
            print("\u2713 Compatible PyTorch installed")

import torch

print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  ({p.total_memory/1024**3:.1f} GB)")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {DEVICE}")

## 1. Clone Repo & Install Dependencies

In [ ]:
REPO_URL = "https://github.com/MRKDaGods/gp.git"
WORK_DIR = Path("/kaggle/working")
PROJECT  = WORK_DIR / "gp"

if not PROJECT.exists():
    print(f"Cloning {REPO_URL} ...")
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT)])
else:
    print("Repo already present, pulling latest ...")
    subprocess.check_call(["git", "-C", str(PROJECT), "pull", "--ff-only"])

os.chdir(str(PROJECT))
sys.path.insert(0, str(PROJECT))
print(f"\n\u2713 Repo ready at {PROJECT}")

In [ ]:
def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

try:
    import faiss; print(f"faiss ok ({faiss.__version__})")
except ImportError:
    try: pip("faiss-gpu")
    except Exception: pip("faiss-cpu")

try:
    import trackeval; print("trackeval ok")
except ImportError:
    pip("git+https://github.com/JonathonLuiten/TrackEval.git")

pip("motmetrics", "loguru", "omegaconf", "rich", "networkx>=3.1", "click",
    "numpy", "scipy", "pandas", "scikit-learn")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps", "-q"],
                      cwd=str(PROJECT))
print("\n\u2713 Dependencies installed")

In [ ]:
FAILED = []
_checks = [
    ("faiss", "faiss"),
    ("motmetrics", "motmetrics"),
    ("loguru", "loguru"),
    ("omegaconf", "omegaconf"),
    ("networkx", "networkx"),
    ("sklearn", "sklearn"),
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("trackeval", "trackeval"),
]
for label, mod in _checks:
    try:
        __import__(mod)
        print(f"  \u2713 {label}")
    except ImportError as e:
        print(f"  \u2717 {label}: {e}")
        FAILED.append(label)
if FAILED:
    raise RuntimeError(f"Missing modules: {FAILED} -- fix pip installs above")
print("\n\u2713 All required modules importable")

## 2. Load Checkpoint from 10b

Finds `checkpoint.tar.gz` in `/kaggle/input/mtmc-10b-stage-3-faiss-indexing/` and extracts it.

In [ ]:
PREV_SLUG = "mtmc-10b-stage-3-faiss-indexing"
PREV_INPUT = Path("/kaggle/input") / PREV_SLUG

if not PREV_INPUT.exists():
    for p in Path("/kaggle/input").iterdir():
        if PREV_SLUG in p.name or "stage3" in p.name or "10b" in p.name:
            PREV_INPUT = p; break

cp = PREV_INPUT / "checkpoint.tar.gz"

# Fallback: download via Kaggle API if not found in mounted input
if not cp.exists():
    print(f"checkpoint.tar.gz not found at {cp} -- attempting API download")
    import subprocess as _sp
    _dl_dir = Path("/tmp/kaggle_dl")
    _dl_dir.mkdir(parents=True, exist_ok=True)
    _r = _sp.run(
        ["kaggle", "kernels", "output",
         "mrkdagods/mtmc-10b-stage-3-faiss-indexing",
         "--file", "checkpoint.tar.gz",
         "-p", str(_dl_dir)],
        capture_output=True, text=True
    )
    print(_r.stdout); print(_r.stderr)
    cp = _dl_dir / "checkpoint.tar.gz"

assert cp.exists(), f"checkpoint.tar.gz not found at {cp}"

EXTRACT_DIR = Path("/tmp/pipeline_run")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Extracting {cp.stat().st_size/1024**2:.1f} MB ...")
with tarfile.open(str(cp), "r:gz") as tar:
    tar.extractall(str(EXTRACT_DIR))

with open(EXTRACT_DIR / "run_metadata.json") as f:
    meta = json.load(f)
RUN_NAME = meta["run_name"]
DATA_OUT = EXTRACT_DIR
RUN_DIR  = EXTRACT_DIR / RUN_NAME
# GT annotations are in the repo under data/raw/cityflowv2/*/gt/gt.txt
# They were force-committed despite data/ being gitignored.
_repo_gt = PROJECT / "data" / "raw" / "cityflowv2"
if any((_repo_gt / cam / "gt" / "gt.txt").exists()
       for cam in ["S01_c001", "S01_c002", "S01_c003",
                   "S02_c006", "S02_c007", "S02_c008"]):
    GT_DIR = str(_repo_gt)
    print(f"\u2713 GT annotations at {GT_DIR}")
else:
    GT_DIR = ""
    print("WARNING: GT files not found in repo — eval will skip metrics")
print(f"\u2713 Checkpoint extracted -- run: {RUN_NAME}")
for s in ["stage1", "stage2", "stage3"]:
    d = RUN_DIR / s
    if d.exists(): print(f"  {s}: {len(list(d.rglob('*')))} files")
print(f"  GT dir: {GT_DIR}")

## 3. Tuning Parameters

**Edit these values** then re-run the cells below (~6 min). No need to re-run 10a or 10b.

In [ ]:
# ---- Stage 4: Cross-camera association -------------------------------------
# v46 optimized config (local best: IDF1=0.8297)
AQE_K             = 2     # v46: 2 (optimal after QE self-exclusion fix)
SIM_THRESH        = 0.53  # v46: 0.53 (optimal from sweep)
ALGORITHM         = "connected_components"
LOUVAIN_RES       = 0.70  # fallback for community_detection

# Weights — v46 defaults (appearance-heavy for vehicles)
APPEARANCE_WEIGHT = 0.75  # v46: CityFlowV2 vehicle config
HSV_WEIGHT        = 0.0   # v46: disabled (hurts vehicles)
ST_WEIGHT         = round(1.0 - APPEARANCE_WEIGHT - HSV_WEIGHT, 4)

# Bridge pruning: v46 found pruning HURTS (-1.4pp) → disabled
BRIDGE_PRUNE      = 0.0   # v46: 0.0 (pruning hurts by -1.4pp)
MAX_COMP_SIZE     = 12

# Intra-camera merge: +1.0pp improvement
INTRA_MERGE       = True
INTRA_MERGE_THRESH = 0.75
INTRA_MERGE_GAP   = 60    # seconds

# v52: Score-level fusion with OSNet secondary embeddings
FUSION_WEIGHT     = 0.1   # v53: 10% secondary (was 0.3 in v52)

# v54: SOTA techniques (AIC21 1st place)
# Camera bias: learn per-camera-pair similarity offsets from clusters
CAMERA_BIAS       = True
CAMERA_BIAS_ITERS = 2     # iterative: cluster → learn bias → re-cluster
# Zone model: penalize impossible transitions, boost valid ones
ZONE_MODEL        = True
ZONE_BONUS        = 0.06  # boost valid zone transitions
ZONE_PENALTY      = 0.04  # penalize invalid transitions
# Hierarchical: multi-pass centroid expansion to recover orphans
HIERARCHICAL      = True
HIER_CENTROID_TH  = 0.35  # orphan → cluster threshold
HIER_MERGE_TH     = 0.35  # cluster ↔ cluster merge threshold
HIER_ORPHAN_TH    = 0.30  # orphan ↔ orphan final threshold

# ---- Stage 5: Evaluation ----------------------------------------------------
# v46: MTMC_ONLY=False keeps single-cam trajectories (+5pp IDF1).
# Single-cam tracks ARE real vehicles; dropping them loses 29/240=12% GT.
MTMC_ONLY = False

print("Stage 4 params (v46 optimized):")
print(f"  aqe_k={AQE_K}  sim_thresh={SIM_THRESH}  algorithm={ALGORITHM}  appearance_weight={APPEARANCE_WEIGHT}")
print(f"  hsv_weight={HSV_WEIGHT}  st_weight={ST_WEIGHT}  bridge_prune={BRIDGE_PRUNE}  max_comp_size={MAX_COMP_SIZE}")
print(f"  intra_merge={INTRA_MERGE}  intra_thresh={INTRA_MERGE_THRESH}  intra_gap={INTRA_MERGE_GAP}")
print(f"  fusion_weight={FUSION_WEIGHT}")
print(f"  camera_bias={CAMERA_BIAS}  zone_model={ZONE_MODEL}  hierarchical={HIERARCHICAL}")
print(f"  zone_bonus={ZONE_BONUS}  zone_penalty={ZONE_PENALTY}")
print(f"  hier: centroid={HIER_CENTROID_TH}  merge={HIER_MERGE_TH}  orphan={HIER_ORPHAN_TH}")
print(f"Stage 5: mtmc_only_submission={MTMC_ONLY}, stationary_filter=True")

## 4. Run Stages 4-5

In [ ]:
os.chdir(str(PROJECT))
cmd = [
    sys.executable, "scripts/run_pipeline.py",
    "--config", "configs/default.yaml",
    "--dataset-config", "configs/datasets/cityflowv2.yaml",
    "--stages", "4,5",
    "--override", f"project.run_name={RUN_NAME}",
    "--override", f"project.output_dir={DATA_OUT}",
    "--override", f"stage4.association.query_expansion.k={AQE_K}",
    "--override", f"stage4.association.graph.similarity_threshold={SIM_THRESH}",
    "--override", f"stage4.association.graph.algorithm={ALGORITHM}",
    "--override", f"stage4.association.graph.louvain_resolution={LOUVAIN_RES}",
    "--override", f"stage4.association.graph.bridge_prune_margin={BRIDGE_PRUNE}",
    "--override", f"stage4.association.graph.max_component_size={MAX_COMP_SIZE}",
    "--override", f"stage4.association.weights.vehicle.appearance={APPEARANCE_WEIGHT}",
    "--override", f"stage4.association.weights.vehicle.hsv={HSV_WEIGHT}",
    "--override", f"stage4.association.weights.vehicle.spatiotemporal={ST_WEIGHT}",
    "--override", "stage4.association.mutual_nn.top_k_per_query=20",
    "--override", "stage4.association.fic.enabled=true",
    "--override", "stage4.association.fic.regularisation=3.0",
    "--override", "stage4.association.fac.enabled=false",
    "--override", "stage4.association.fac.knn=20",
    "--override", "stage4.association.fac.learning_rate=0.5",
    "--override", "stage4.association.fac.beta=0.08",
    # v52: score-level fusion with OSNet secondary embeddings
    "--override", f"stage4.association.secondary_embeddings.path={RUN_DIR}/stage2/embeddings_secondary.npy",
    "--override", f"stage4.association.secondary_embeddings.weight={FUSION_WEIGHT}",
    # v54: SOTA — camera distance bias
    "--override", f"stage4.association.camera_bias.enabled={str(CAMERA_BIAS).lower()}",
    "--override", f"stage4.association.camera_bias.iterations={CAMERA_BIAS_ITERS}",
    # v54: SOTA — zone-based transition scoring
    "--override", f"stage4.association.zone_model.enabled={str(ZONE_MODEL).lower()}",
    "--override", "stage4.association.zone_model.zone_data_path=configs/datasets/cityflowv2_zones.json",
    "--override", f"stage4.association.zone_model.bonus={ZONE_BONUS}",
    "--override", f"stage4.association.zone_model.penalty={ZONE_PENALTY}",
    # v54: SOTA — hierarchical centroid expansion
    "--override", f"stage4.association.hierarchical.enabled={str(HIERARCHICAL).lower()}",
    "--override", f"stage4.association.hierarchical.centroid_threshold={HIER_CENTROID_TH}",
    "--override", f"stage4.association.hierarchical.merge_threshold={HIER_MERGE_TH}",
    "--override", f"stage4.association.hierarchical.orphan_threshold={HIER_ORPHAN_TH}",
    "--override", "stage4.association.hierarchical.max_merge_size=12",
    "--override", f"stage4.association.intra_camera_merge.enabled={str(INTRA_MERGE).lower()}",
    "--override", f"stage4.association.intra_camera_merge.threshold={INTRA_MERGE_THRESH}",
    "--override", f"stage4.association.intra_camera_merge.max_time_gap={INTRA_MERGE_GAP}",
    "--override", f"stage5.mtmc_only_submission={str(MTMC_ONLY).lower()}",
    "--override", "stage5.stationary_filter.enabled=true",
    "--override", "stage5.stationary_filter.min_displacement_px=150",
    "--override", "stage5.stationary_filter.max_mean_velocity_px=2.0",
    "--override", "stage5.min_submission_confidence=0.15",
    "--override", "stage5.cross_id_nms_iou=0.35",
    "--override", "stage5.min_trajectory_confidence=0.30",
    "--override", "stage5.min_trajectory_frames=10",
    "--override", "stage5.track_edge_trim.enabled=false",
    "--override", "stage5.track_smoothing.enabled=false",
    "--override", "stage5.gt_frame_clip=true",
    "--override", "stage5.gt_zone_filter=true",
]
if GT_DIR:
    cmd += ["--override", f"stage5.ground_truth_dir={GT_DIR}"]
else:
    print("WARNING: GT_DIR is empty — eval will skip metric computation")
print("CMD:", " ".join(str(c) for c in cmd))
print("=" * 70)
t0 = time.time()
r = subprocess.run(cmd, cwd=str(PROJECT))
print("=" * 70)
elapsed = time.time() - t0
if r.returncode != 0:
    print(f"\u2717 FAILED after {elapsed/60:.1f} min"); sys.exit(r.returncode)
print(f"\u2713 Stages 4-5 done in {elapsed/60:.1f} min")

## 5. Results

In [ ]:
stage5_dir = RUN_DIR / "stage5"

def _pct(v):
    return f"{v:.1%}" if isinstance(v, float) else str(v)

metrics_files = sorted(stage5_dir.glob("metrics_*.json")) if stage5_dir.exists() else []
if metrics_files:
    print("=" * 65)
    print("EVALUATION RESULTS")
    print("=" * 65)
    for mf in metrics_files:
        m = json.loads(mf.read_text())
        m = m.get("metrics", m)
        cam = mf.stem.replace("metrics_", "")
        idf1 = _pct(m.get("IDF1", m.get("idf1", "?")))
        mota = _pct(m.get("MOTA", m.get("mota", "?")))
        hota = _pct(m.get("HOTA", m.get("hota", "?")))
        idsw = m.get("ID_Sw", m.get("id_switches", "?"))
        print(f"  {cam:12s}  IDF1={idf1}  MOTA={mota}  HOTA={hota}  IDsw={idsw}")

for fname in ["summary.json", "evaluation_report.json"]:
    sf = stage5_dir / fname
    if sf.exists():
        s = json.loads(sf.read_text())
        print("-" * 65 + "\n  GLOBAL:")
        for k in ["IDF1","MOTA","HOTA","ID_Sw","idf1","mota","hota","id_switches","mtmc_idf1"]:
            v = s.get(k)
            if v is not None: print(f"    {k}: {_pct(v)}")
        break

if not metrics_files:
    print("No metrics files found -- check stage5 output dir:", stage5_dir)

# Copy evaluation report to /kaggle/working/ for easy download
import shutil as _shutil
for _fname in ["evaluation_report.json", "summary.json"]:
    _src = stage5_dir / _fname
    if _src.exists():
        _shutil.copy2(str(_src), str(Path("/kaggle/working") / _fname))
        print(f"Copied {_fname} to /kaggle/working/")


## 6. Automated Parameter Scan (optional)

Runs stages 4-5 across a grid of parameter values and reports the best combination.
Each combination takes ~2 min → a 12-point scan takes ~24 min total.

**Comment out this cell if you just want the single run above.**

In [ ]:
# ============================================================
# SET SCAN_ENABLED = False to run the grid search.
# This will overwrite the single-run results above.
# ============================================================
SCAN_ENABLED = False

if SCAN_ENABLED:
    import itertools

    # Grid to search — v14: updated after v12 scan analysis.
    # v12 finding: CC and CD produce identical results → removed algorithm axis.
    # Added gallery_expansion.threshold to scan orphan recovery aggressiveness.
    # v15: added length_weight_power (0.0=no penalty, 0.5=penalize short tracklets)
    # v16: added orphan_match_threshold to test Phase 2 orphan-orphan matching
    # v18: added reranking lambda (0=disabled, >0=enabled with that lambda).
    #       Dropped len_weight axis (fix at 0.5) to keep grid <400.
    #       Added cross-ID NMS in format_converter.
    # v19: replaced appearance_w=0.65 with 0.95 (near-zero ST) because
    #       ST score is near-constant for CityFlowV2 (all pairs 0.7-1.0) →
    #       27.5% of combined score was noise. test if appearance-dominant is better.
    #       Orphan matching now uses GraphSolver with bridge pruning.
    #       Reranking uses pre-computed sim matrix for ~10x speedup.
    # v22: AIC21 SOTA uses very strict thresholds for sub-clustering.
    #      Added 0.60 and 0.65 to explore more conservative matching.
    #      FIC+FAC enabled, reranking lambda from SOTA.
    #      Reduced other axes to keep combos manageable.
    # Total: 6 × 2 × 2 × 2 × 2 × 2 × 2 = 384 combos (~160 min at ~25s each)
    scan_grid = {
        "sim_thresh":       [0.35, 0.40, 0.50, 0.55, 0.60, 0.65],  # 6: wider range incl. conservative
        "appearance_w":     [0.80, 0.95],                     # 2
        "bridge_prune":     [0.0, 0.05],                      # 2: bridge pruning on/off
        "gallery_thresh":   [0.40, 0.55],                     # 2: reduced from 3 (0.35 was rarely best)
        "orphan_thresh":    [0.0, 0.30],                      # 2: Phase 2 orphan-orphan (0=disabled)
        "rerank_lambda":    [0.0, 0.3],                       # 2: 0=off, 0.3=AIC21 SOTA k-reciprocal blending
        "algorithm":        ["connected_components", "agglomerative"],  # 2: v20: AIC21 uses agglomerative
    }
    HSV_W_FIXED = 0.025  # v11: lowered to match reference

    keys   = list(scan_grid.keys())
    combos = list(itertools.product(*[scan_grid[k] for k in keys]))
    print(f"Scanning {len(combos)} combinations ...")

    results = []
    for combo in combos:
        params = dict(zip(keys, combo))
        # Build scan run name from all parameters
        bridge_tag = f"bp{params['bridge_prune']:.2f}".replace(".", "")
        app_tag = f"app{params['appearance_w']:.2f}".replace(".", "")
        gal_tag = f"gal{params['gallery_thresh']:.2f}".replace(".", "")
        ot_tag = f"ot{params['orphan_thresh']:.2f}".replace(".", "")
        rr_tag = f"rr{params['rerank_lambda']:.1f}".replace(".", "")
        alg_tag = "agg" if params["algorithm"] == "agglomerative" else "cc"
        scan_run = f"scan_{params['sim_thresh']}_{app_tag}_{bridge_tag}_{gal_tag}_{ot_tag}_{rr_tag}_{alg_tag}"

        # Stage 4 reads stage1/stage2/stage3 from output_base/run_name/.
        # Symlink the upstream outputs so the scan sub-dir looks like a full run.
        scan_dir = DATA_OUT / scan_run
        scan_dir.mkdir(parents=True, exist_ok=True)
        for stage_sub in ("stage1", "stage2", "stage3"):
            src = DATA_OUT / RUN_NAME / stage_sub
            dst = scan_dir / stage_sub
            if src.exists() and not dst.exists():
                dst.symlink_to(src)

        # Compute spatiotemporal weight to maintain sum=1.0
        # hsv is fixed; st = 1 - appearance - hsv
        st_w = round(1.0 - params["appearance_w"] - HSV_W_FIXED, 4)

        # Determine reranking enabled state from lambda value
        rerank_enabled = params["rerank_lambda"] > 0

        cmd_scan = [
            sys.executable, "scripts/run_pipeline.py",
            "--config", "configs/default.yaml",
            "--dataset-config", "configs/datasets/cityflowv2.yaml",
            "--stages", "4,5",
            "--override", f"project.run_name={scan_run}",
            "--override", f"project.output_dir={DATA_OUT}",
            "--override", f"stage4.association.query_expansion.k={AQE_K}",
            "--override", f"stage4.association.graph.similarity_threshold={params['sim_thresh']}",
            "--override", f"stage4.association.graph.bridge_prune_margin={params['bridge_prune']}",
            "--override", f"stage4.association.graph.algorithm={params['algorithm']}",
            "--override", f"stage4.association.graph.louvain_resolution={LOUVAIN_RES}",
            "--override", f"stage4.association.graph.max_component_size={MAX_COMP_SIZE}",
            "--override", f"stage4.association.gallery_expansion.threshold={params['gallery_thresh']}",
            "--override", f"stage4.association.weights.vehicle.appearance={params['appearance_w']}",
            "--override", f"stage4.association.weights.vehicle.hsv={HSV_W_FIXED}",
            "--override", f"stage4.association.weights.vehicle.spatiotemporal={st_w}",
            "--override", "stage4.association.weights.length_weight_power=0.5",
            "--override", f"stage4.association.gallery_expansion.orphan_match_threshold={params['orphan_thresh']}",
            "--override", "stage4.association.mutual_nn.top_k_per_query=20",
            "--override", "stage4.association.fic.enabled=true",
            "--override", "stage4.association.fic.regularisation=3.0",
            "--override", "stage4.association.fac.enabled=false",
            "--override", "stage4.association.fac.knn=20",
            "--override", "stage4.association.fac.learning_rate=0.5",
            "--override", "stage4.association.fac.beta=0.08",
            "--override", f"stage4.association.intra_camera_merge.enabled={str(INTRA_MERGE).lower()}",
            "--override", f"stage4.association.intra_camera_merge.threshold={INTRA_MERGE_THRESH}",
            "--override", f"stage4.association.intra_camera_merge.max_time_gap={INTRA_MERGE_GAP}",
            "--override", f"stage4.association.reranking.enabled={str(rerank_enabled).lower()}",
            "--override", f"stage4.association.reranking.lambda_value={params['rerank_lambda']}",
            "--override", f"stage5.mtmc_only_submission={str(MTMC_ONLY).lower()}",
            "--override", "stage5.stationary_filter.enabled=true",
            "--override", "stage5.stationary_filter.min_displacement_px=150",
            "--override", "stage5.stationary_filter.max_mean_velocity_px=2.0",
            "--override", "stage5.min_submission_confidence=0.15",
            "--override", "stage5.cross_id_nms_iou=0.35",
            "--override", "stage5.min_trajectory_confidence=0.30",
            "--override", "stage5.min_trajectory_frames=10",
            "--override", "stage5.track_edge_trim.enabled=false",
            "--override", "stage5.track_smoothing.enabled=false",
            "--override", "stage5.gt_frame_clip=true",
            "--override", "stage5.gt_zone_filter=true",
        ]
        if GT_DIR:
            cmd_scan += ["--override", f"stage5.ground_truth_dir={GT_DIR}"]
        t0 = time.time()
        r = subprocess.run(cmd_scan, cwd=str(PROJECT), capture_output=True)
        elapsed = time.time() - t0

        # Read metric from evaluation report
        report = DATA_OUT / scan_run / "stage5" / "evaluation_report.json"
        idf1 = mota = hota = mtmc_mota = 0.0
        if report.exists():
            rp = json.loads(report.read_text())
            m = rp.get("metrics", rp)
            idf1 = m.get("mtmc_idf1", m.get("IDF1", m.get("idf1", 0.0)))
            mota = m.get("MOTA", m.get("mota", 0.0))
            hota = m.get("HOTA", m.get("hota", 0.0))
            details = rp.get("details", {})
            mtmc_mota = details.get("mtmc_mota", mota)

        results.append({**params, "st_w": st_w, "IDF1": idf1, "MOTA": mota, "MTMC_MOTA": mtmc_mota, "HOTA": hota, "time": elapsed})
        status = "OK" if r.returncode == 0 else "FAIL"
        print(f"  [{status}] {params} st_w={st_w:.2f} -> IDF1={idf1:.3f} MOTA={mota:.3f} HOTA={hota:.3f} ({elapsed/60:.1f} min)")

    # Sort by HOTA when > 0 (HOTA is more robust), fallback to MTMC IDF1
    has_hota = any(r2["HOTA"] > 0 for r2 in results)
    sort_key = "HOTA" if has_hota else "IDF1"
    results.sort(key=lambda x: x[sort_key], reverse=True)
    print("\n" + "=" * 80)
    print(f"SCAN RESULTS (sorted by {sort_key})")
    print("=" * 80)
    header = f"{'sim':<6} {'app_w':<7} {'bridge':<8} {'gal_th':<7} {'orph':<6} {'rr_λ':<6} {'alg':<5} {'st_w':<7} {'IDF1':>7} {'MOTA':>7} {'HOTA':>7}"
    print(header)
    for r2 in results:
        alg_short = 'agg' if r2.get('algorithm','') == 'agglomerative' else 'cc'
        print(f"{r2['sim_thresh']:<6} {r2['appearance_w']:<7} "
              f"{r2['bridge_prune']:<8} {r2.get('gallery_thresh','?'):<7} {r2.get('orphan_thresh',0.30):<6} {r2.get('rerank_lambda',0.0):<6} {alg_short:<5} {r2.get('st_w',0.25):<7} "
              f"{r2['IDF1']:>7.3f} {r2['MOTA']:>7.3f} {r2['HOTA']:>7.3f}")
    best = results[0]
    print(f"\nBEST: sim={best['sim_thresh']} app={best['appearance_w']} "
          f"bridge={best['bridge_prune']} gal={best.get('gallery_thresh','?')} "
          f"orph={best.get('orphan_thresh',0.30)} rr_lambda={best.get('rerank_lambda',0.0)} "
          f"alg={best.get('algorithm','?')} "
          f"-> IDF1={best['IDF1']:.3f} MOTA={best['MOTA']:.3f} HOTA={best['HOTA']:.3f}")

    # ── Parameter sensitivity analysis ──
    print("\n" + "=" * 80)
    print("PARAMETER SENSITIVITY ANALYSIS")
    print("=" * 80)
    for param_name in scan_grid:
        param_vals = sorted(set(r2[param_name] for r2 in results))
        if len(param_vals) < 2:
            continue
        print(f"\n--- {param_name} ---")
        for pval in param_vals:
            subset = [r2 for r2 in results if r2[param_name] == pval]
            avg_hota = sum(r2["HOTA"] for r2 in subset) / len(subset) if subset else 0
            avg_idf1 = sum(r2["IDF1"] for r2 in subset) / len(subset) if subset else 0
            avg_mota = sum(r2["MOTA"] for r2 in subset) / len(subset) if subset else 0
            best_hota = max(r2["HOTA"] for r2 in subset) if subset else 0
            print(f"  {param_name}={pval:<10} avg HOTA={avg_hota:.3f} avg IDF1={avg_idf1:.3f} "
                  f"avg MOTA={avg_mota:.3f} best HOTA={best_hota:.3f} (n={len(subset)})")

    # ── MTMC MOTA breakdown ──
    if any(r2.get("MTMC_MOTA", 0) != 0 for r2 in results):
        print("\n" + "=" * 80)
        print("TOP 10 by MTMC_MOTA:")
        by_mtmc = sorted(results, key=lambda x: x.get("MTMC_MOTA", 0), reverse=True)
        for r2 in by_mtmc[:10]:
            alg_short = 'agg' if r2.get('algorithm','') == 'agglomerative' else 'cc'
            print(f"  sim={r2['sim_thresh']} app={r2['appearance_w']} alg={alg_short} "
                  f"rr={r2.get('rerank_lambda',0)} -> MTMC_MOTA={r2.get('MTMC_MOTA',0):.3f} "
                  f"IDF1={r2['IDF1']:.3f} HOTA={r2['HOTA']:.3f}")
    # Save results to JSON for offline analysis
    import json as _json
    results_path = DATA_OUT / "scan_results.json"
    with open(results_path, "w") as _f:
        _json.dump({"sort_key": sort_key, "results": results}, _f, indent=2)
    print(f"Scan results saved to {results_path}")

    # Copy results to /kaggle/working/ so `kaggle kernels output --file` can download them
    import shutil as _shutil
    _out = Path("/kaggle/working")
    _shutil.copy2(str(results_path), str(_out / "scan_results.json"))
    print(f"Copied scan_results.json to {_out}")
    # Also copy the best run's evaluation report
    best_bridge_tag = f"bp{best['bridge_prune']:.2f}".replace(".", "")
    best_app_tag = f"app{best['appearance_w']:.2f}".replace(".", "")
    best_gal_tag = f"gal{best.get('gallery_thresh', 0.45):.2f}".replace(".", "")
    best_ot_tag = f"ot{best.get('orphan_thresh', 0.30):.2f}".replace(".", "")
    best_rr_tag = f"rr{best.get('rerank_lambda', 0.0):.1f}".replace(".", "")
    best_alg_tag = "agg" if best.get("algorithm", "") == "agglomerative" else "cc"
    best_run = f"scan_{best['sim_thresh']}_{best_app_tag}_{best_bridge_tag}_{best_gal_tag}_{best_ot_tag}_{best_rr_tag}_{best_alg_tag}"
    best_report = DATA_OUT / best_run / "stage5" / "evaluation_report.json"
    if best_report.exists():
        _shutil.copy2(str(best_report), str(_out / "evaluation_report_best.json"))
        print(f"Copied best evaluation report to {_out}")
else:
    print("Scan disabled. Set SCAN_ENABLED = False to run grid search.")

## 7. Feature A/B Tests

Tests untried association features vs the v46 baseline.
- **CSLS**: Cross-domain Similarity Local Scaling — hubness reduction to penalize 'universal hub' embeddings

In [ ]:
# --- Feature A/B tests ---
# Tests CSLS hubness reduction (last untested stage4 feature)
FEATURE_TEST_ENABLED = False

if FEATURE_TEST_ENABLED:
    # Define experiments: (tag, extra_overrides_dict)
    feature_experiments = [
        # --- temporal_split: split clusters at time gaps (targets conflation) ---
        ("tsplit_gap60_t050", {"stage4.association.temporal_split.enabled": "true",
                               "stage4.association.temporal_split.min_gap": "60",
                               "stage4.association.temporal_split.split_threshold": "0.50"}),
        ("tsplit_gap45_t045", {"stage4.association.temporal_split.enabled": "true",
                               "stage4.association.temporal_split.min_gap": "45",
                               "stage4.association.temporal_split.split_threshold": "0.45"}),
        ("tsplit_gap30_t040", {"stage4.association.temporal_split.enabled": "true",
                               "stage4.association.temporal_split.min_gap": "30",
                               "stage4.association.temporal_split.split_threshold": "0.40"}),
        # --- cluster_verify: eject weakly-connected cluster members ---
        ("cverify_030", {"stage4.association.cluster_verify.enabled": "true",
                         "stage4.association.cluster_verify.min_connectivity": "0.30"}),
        ("cverify_035", {"stage4.association.cluster_verify.enabled": "true",
                         "stage4.association.cluster_verify.min_connectivity": "0.35"}),
        ("cverify_025", {"stage4.association.cluster_verify.enabled": "true",
                         "stage4.association.cluster_verify.min_connectivity": "0.25"}),
        # --- combined: temporal_split + cluster_verify ---
        ("tsplit60_cverify030", {"stage4.association.temporal_split.enabled": "true",
                                 "stage4.association.temporal_split.min_gap": "60",
                                 "stage4.association.temporal_split.split_threshold": "0.50",
                                 "stage4.association.cluster_verify.enabled": "true",
                                 "stage4.association.cluster_verify.min_connectivity": "0.30"}),
    ]
    feat_results = []

    for tag, extra_overrides in feature_experiments:
        feat_run = f"{RUN_NAME}_{tag}"
        feat_dir = DATA_OUT / feat_run
        feat_dir.mkdir(parents=True, exist_ok=True)
        for stage_sub in ("stage1", "stage2", "stage3"):
            src = RUN_DIR / stage_sub
            dst = feat_dir / stage_sub
            if src.exists() and not dst.exists():
                dst.symlink_to(src)

        st_w = round(1.0 - APPEARANCE_WEIGHT - HSV_WEIGHT, 4)
        cmd_feat = [
            sys.executable, "scripts/run_pipeline.py",
            "--config", "configs/default.yaml",
            "--dataset-config", "configs/datasets/cityflowv2.yaml",
            "--stages", "4,5",
            "--override", f"project.run_name={feat_run}",
            "--override", f"project.output_dir={DATA_OUT}",
            "--override", f"stage4.association.query_expansion.k={AQE_K}",
            "--override", f"stage4.association.graph.similarity_threshold={SIM_THRESH}",
            "--override", f"stage4.association.graph.algorithm={ALGORITHM}",
            "--override", f"stage4.association.graph.bridge_prune_margin={BRIDGE_PRUNE}",
            "--override", f"stage4.association.graph.max_component_size={MAX_COMP_SIZE}",
            "--override", f"stage4.association.weights.vehicle.appearance={APPEARANCE_WEIGHT}",
            "--override", f"stage4.association.weights.vehicle.hsv={HSV_WEIGHT}",
            "--override", f"stage4.association.weights.vehicle.spatiotemporal={st_w}",
            "--override", "stage4.association.weights.length_weight_power=0.5",
            "--override", "stage4.association.mutual_nn.top_k_per_query=20",
            "--override", "stage4.association.fic.enabled=true",
            "--override", "stage4.association.fic.regularisation=3.0",
            "--override", "stage4.association.fac.enabled=false",
            "--override", "stage4.association.fac.knn=20",
            "--override", "stage4.association.fac.learning_rate=0.5",
            "--override", "stage4.association.fac.beta=0.08",
            "--override", f"stage4.association.intra_camera_merge.enabled={str(INTRA_MERGE).lower()}",
            "--override", f"stage4.association.intra_camera_merge.threshold={INTRA_MERGE_THRESH}",
            "--override", f"stage4.association.intra_camera_merge.max_time_gap={INTRA_MERGE_GAP}",
            "--override", "stage4.association.gallery_expansion.enabled=true",
            "--override", f"stage5.mtmc_only_submission={str(MTMC_ONLY).lower()}",
            "--override", "stage5.stationary_filter.enabled=true",
            "--override", "stage5.stationary_filter.min_displacement_px=150",
            "--override", "stage5.stationary_filter.max_mean_velocity_px=2.0",
            "--override", "stage5.min_submission_confidence=0.15",
            "--override", "stage5.cross_id_nms_iou=0.35",
            "--override", "stage5.min_trajectory_confidence=0.30",
            "--override", "stage5.min_trajectory_frames=10",
            "--override", "stage5.track_edge_trim.enabled=false",
            "--override", "stage5.track_smoothing.enabled=false",
            "--override", "stage5.gt_frame_clip=true",
            "--override", "stage5.gt_zone_filter=true",
        ]
        # Add feature-specific overrides
        for k, v in extra_overrides.items():
            cmd_feat += ["--override", f"{k}={v}"]
        if GT_DIR:
            cmd_feat += ["--override", f"stage5.ground_truth_dir={GT_DIR}"]

        t0 = time.time()
        r = subprocess.run(cmd_feat, cwd=str(PROJECT), capture_output=True)
        elapsed = time.time() - t0

        report = DATA_OUT / feat_run / "stage5" / "evaluation_report.json"
        idf1 = mota = hota = mtmc_idf1 = 0.0
        n_frag = n_conf = 0
        if report.exists():
            rp = json.loads(report.read_text())
            m = rp.get("metrics", rp)
            idf1 = m.get("IDF1", m.get("idf1", 0.0))
            mota = m.get("MOTA", m.get("mota", 0.0))
            hota = m.get("HOTA", m.get("hota", 0.0))
            mtmc_idf1 = m.get("mtmc_idf1", idf1)
            details = rp.get("details", {})
            n_frag = details.get("n_fragmented_gt", 0)
            n_conf = details.get("n_conflated_pred", 0)
        status = "OK" if r.returncode == 0 else "FAIL"
        feat_results.append({"tag": tag, "IDF1": idf1, "mtmc_idf1": mtmc_idf1,
                             "MOTA": mota, "HOTA": hota, "frag": n_frag, "conf": n_conf, "time": elapsed})
        print(f"  [{status}] {tag}: IDF1={idf1:.3f} mtmc_idf1={mtmc_idf1:.3f} MOTA={mota:.3f} HOTA={hota:.3f} frag={n_frag} conf={n_conf} ({elapsed/60:.1f} min)")

    print("\n" + "=" * 80)
    print("FEATURE A/B TEST RESULTS")
    print("=" * 80)
    print(f"  {'tag':<25} {'IDF1':>7} {'mtmc':>7} {'MOTA':>7} {'HOTA':>7} {'frag':>5} {'conf':>5}")
    print(f"  {'baseline (v46)' :<25} {'---':>7} {'---':>7} {'---':>7} {'---':>7} {'---':>5} {'---':>5}  (see section 5)")
    for r2 in sorted(feat_results, key=lambda x: x["IDF1"], reverse=True):
        print(f"  {r2['tag']:<25} {r2['IDF1']:>7.3f} {r2['mtmc_idf1']:>7.3f} {r2['MOTA']:>7.3f} {r2['HOTA']:>7.3f} {r2['frag']:>5} {r2['conf']:>5}")
else:
    print("Feature test disabled. Set FEATURE_TEST_ENABLED = False to run.")